In [0]:
# pipeline_ebird_bronze.py  (DLT pipeline notebook)
#
# Attach this as a Delta Live Tables pipeline in Databricks, not a regular job.
# Auto Loader watches the Volume paths and processes new files as they arrive.
#
# Pipeline settings (set in the DLT UI or pipeline JSON):
#   catalog  : birds
#   target   : ebird_bronze
#   channel  : current

import dlt
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, TimestampType

VOLUME_ROOT = "/Volumes/birds/ebird_landing/raw"

# ── Bronze: checklist_index ───────────────────────────────────────────────────
@dlt.table(
    name="checklist_index",
    comment="Raw eBird checklist index — one row per checklist stub, payload intact",
    table_properties={
        "quality":                "bronze",
        "pipelines.autoOptimize.managed": "true",
    },
    partition_cols=["_run_date"],
)
@dlt.expect("envelope_present", "payload IS NOT NULL")
def checklist_index_bronze():
    return (
        spark.readStream
             .format("cloudFiles")
             .option("cloudFiles.format",         "json")
             .option("cloudFiles.inferColumnTypes", "false")  # never infer in bronze
             # Schema location persists Auto Loader's inferred schema across restarts.
             # Stored in the volume alongside the data.
             .option("cloudFiles.schemaLocation",
                     f"{VOLUME_ROOT}/checklist_index/_schema_hint")
             .load(f"{VOLUME_ROOT}/checklist_index/")
             # Promote the envelope fields Auto Loader reads as top-level columns
             .select(
                 F.col("_run_id"),
                 F.col("_ingestion_ts").cast(TimestampType()),
                 F.col("_run_date"),
                 F.col("_source"),
                 F.col("_page_num").cast(IntegerType()),
                 F.col("payload"),                     # full struct — untouched
                 F.to_json(F.col("payload")).alias("raw_json"),  # string copy for auditability
                 # Auto Loader metadata — useful for debugging bad files
                 F.col("_metadata.file_path").alias("_source_file"),
                 F.col("_metadata.file_modification_time").alias("_file_modified_ts"),
             )
    )


# ── Bronze: checklist_detail ──────────────────────────────────────────────────
@dlt.table(
    name="checklist_detail",
    comment="Raw eBird checklist detail — one row per checklist, payload intact. Error rows included.",
    table_properties={
        "quality":                "bronze",
        "pipelines.autoOptimize.managed": "true",
    },
    partition_cols=["_run_date"],
)
@dlt.expect("sub_id_present", "payload.subId IS NOT NULL OR _fetch_error IS NOT NULL")
def checklist_detail_bronze():
    return (
        spark.readStream
             .format("cloudFiles")
             .option("cloudFiles.format",          "json")
             .option("cloudFiles.inferColumnTypes", "false")
             .option("cloudFiles.schemaLocation",
                     f"{VOLUME_ROOT}/checklist_detail/_schema_hint")
             .load(f"{VOLUME_ROOT}/checklist_detail/")
             .select(
                 F.col("_run_id"),
                 F.col("_ingestion_ts").cast(TimestampType()),
                 F.col("_run_date"),
                 F.col("_source"),
                 F.col("_http_status").cast(IntegerType()),
                 F.col("_fetch_error"),
                 F.col("payload"),
                 F.to_json(F.col("payload")).alias("raw_json"),
                 F.col("_metadata.file_path").alias("_source_file"),
                 F.col("_metadata.file_modification_time").alias("_file_modified_ts"),
             )
    )